# Exploración del Chilean Spanish Corpus (base del paso 1)

Antes de entrenar Gemma 4 E4B con predicción del siguiente token (NTP) sobre el corpus, este notebook responde:

1. **Qué fuentes tiene el corpus y cuánto pesa cada una**, en filas y en tokens de Gemma.
2. **Qué hay que limpiar**: URLs, menciones, retweets, duplicados, textos cortos o sin letras.
3. **Qué tan chilena y coloquial es cada fuente**, según marcadores de habla chilena y las expresiones de nuestro diccionario.
4. **Cuántas expresiones de nuestro diccionario (`rag_documentos.jsonl`) aparecen en el corpus**, contando sobre el corpus y no sobre la muestra, incluidas las del dataset de traducción (paso 2) por split.

El corpus tiene ~37M textos (~19 GB en 63 parquets), así que **no se descarga completo**. Se leen los metadatos de cada parquet para armar un índice *(archivo, row group, fuente)* y con él se toma una muestra estratificada por fuente. Al final se guarda `config_paso1.json` (mezcla de fuentes y filtros), que lee `01_paso1_ntp_corpus.ipynb`.

**Requisitos**
- El dataset tiene acceso restringido: acepta sus condiciones en https://huggingface.co/datasets/jorgeortizfuentes/chilean-spanish-corpus y guarda un token de lectura como secreto `HF_TOKEN` en Colab (el ícono de la llave).
- **No necesita GPU**: basta un entorno de CPU, idealmente con RAM alta.

In [ ]:
%pip install -q -U "transformers>=5.5" "peft>=0.17" accelerate datasets huggingface_hub sacrebleu
%pip uninstall -q -y torchao   # el torchao de Colab (0.10) rompe a peft al crear el LoRA; no se usa pyahocorasick

In [ ]:
# Configuración
REPO_HF = "mariopedemonte12/chineutro-gemma4"   # repo privado con datos, checkpoints y resultados; None = solo local
DIR_TRABAJO = None   # None = salidas/gemma4 dentro del repo git, o /content/chineutro en Colab
N_POR_FUENTE = 20_000   # textos por fuente en la muestra
SEED = 42
MIN_PALABRAS = 5        # filtros que se proponen para el paso 1
MAX_CARACTERES = 4000   # los documentos web largos se recortan para que no dominen la mezcla

### Datos, checkpoints y resultados: repo privado del Hugging Face Hub

Todo vive en el repo privado `REPO_HF`, que es un espejo de `DIR_TRABAJO`: `DIR_TRABAJO/x/y` queda en `x/y` del repo.

| Ruta en el repo | Contenido |
|---|---|
| `datos/` | `comun.py`, `lanzar.py`, los notebooks, `rag_documentos.jsonl`, `dataset_chilenismos_opus.csv`, `split_oficial.csv` |
| `indice_corpus.parquet`, `config_paso1.json`, `muestra_*.parquet`, `presencia_diccionario_*.csv` | salidas de la exploración (00) |
| `paso1/corpus_<M>M.parquet` | corpus del paso 1 ya construido |
| `paso1/<nombre>/last-checkpoint/` | último checkpoint del paso 1 (se reemplaza en cada guardado) |
| `paso1/<nombre>/adaptador/`, `metricas_paso1.json` | adaptador final y métricas |
| `paso2/<condición>/` | adaptador, predicciones y métricas de cada condición |
| `ejecuciones/` | notebook ejecutado y log de cada corrida sin interfaz (`lanzar.py`) |

**Sin interfaz** (runtime de Colab por API): `python datos/lanzar.py <notebook> --repo REPO_HF -p CONDICION paso1+2`. Corre el notebook con papermill y sube el log cada 10 minutos.

**Dentro del repo git** (local) se usan los datos del repo y `DIR_TRABAJO = salidas/gemma4`. Con `REPO_HF = None` no se sube ni se baja nada.

El token de Hugging Face (variable `HF_TOKEN`, `hf auth login` o el secreto `HF_TOKEN` de Colab) necesita **permiso de escritura** para subir resultados, y acceso al corpus restringido.

In [ ]:
# Entorno: token de Hugging Face, comun.py y los datos, y el repo del Hub
import os, sys, json
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

# Sin interfaz no se puede pedir el token: tiene que estar en HF_TOKEN, en el archivo de `hf auth login`
# o en los secretos de Colab
from huggingface_hub import get_token, snapshot_download
if not get_token() and EN_COLAB:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert get_token(), "Falta el token de Hugging Face (HF_TOKEN o hf auth login)"


def raiz_repo():
    """Raíz del repo git, si el notebook corre dentro de él."""
    actual = Path.cwd().resolve()
    return next((c for c in [actual, *actual.parents] if (c / "models/gemma4/comun.py").is_file()), None)


RAIZ_REPO = raiz_repo()
if DIR_TRABAJO is None:
    DIR_TRABAJO = RAIZ_REPO / "salidas/gemma4" if RAIZ_REPO else "/content/chineutro"
DIR_TRABAJO = Path(DIR_TRABAJO)
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)

if RAIZ_REPO:   # dentro del repo git: datos y código del repo
    RAIZ, DIR_CODIGO = RAIZ_REPO, RAIZ_REPO / "models/gemma4"
else:           # Colab: comun.py y los datos vienen de datos/ en el repo del Hub
    if REPO_HF:
        snapshot_download(REPO_HF, allow_patterns=["datos/*"], local_dir=DIR_TRABAJO)
    RAIZ = DIR_CODIGO = DIR_TRABAJO / "datos"
    assert (DIR_CODIGO / "comun.py").is_file(), f"No encontré comun.py en {DIR_CODIGO}"
sys.path.insert(0, str(DIR_CODIGO))
import comun

HUB = comun.Hub(REPO_HF, DIR_TRABAJO)   # REPO_HF = None: no sube ni baja nada
for relativa in (comun.RUTA_RAG, comun.RUTA_PARES, comun.RUTA_SPLIT):
    try:
        print("✓", comun.ruta_dato(RAIZ, relativa))
    except FileNotFoundError as e:
        print(f"✗ {e}")
print("Datos:", RAIZ, "| salidas:", DIR_TRABAJO, "| Hub:", REPO_HF or "desactivado")

In [ ]:
# Estilo de gráficos: una serie en azul, grilla y ejes discretos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AZUL, NARANJA, TINTA, TINTA_2, GRILLA = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.edgecolor": GRILLA, "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2,
                     "ytick.color": TINTA_2, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRILLA, "grid.linewidth": 0.6, "axes.axisbelow": True,
                     "font.size": 10, "axes.titlesize": 11})
pd.set_option("display.max_colwidth", 200)


def barras(serie, titulo, xlabel, log=False, fmt="{:,.0f}"):
    """Una serie por categoría. En escala log se dibujan puntos: el largo de una barra en log engaña."""
    s = serie.sort_values()
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(s) + 1.2))
    if log:
        ax.plot(s.values, s.index.astype(str), "o", color=AZUL, markersize=8)
        ax.set_xscale("log")
        ax.grid(axis="y", visible=True)
    else:
        ax.barh(s.index.astype(str), s.values, color=AZUL, height=0.6)
        ax.grid(axis="y", visible=False)
    for y, v in enumerate(s.values):
        ax.text(v, y, " " + fmt.format(v), va="center", color=TINTA_2, fontsize=9)
    ax.set_title(titulo, loc="left", color=TINTA)
    ax.set_xlabel(xlabel)
    plt.tight_layout()
    plt.show()


def curva_perdida(historial, titulo, ylabel, guardar=None):
    """Pérdida de entrenamiento (línea) y de validación (puntos) desde trainer.state.log_history."""
    hist = pd.DataFrame(historial)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    if "loss" in hist:
        tr = hist.dropna(subset=["loss"])
        ax.plot(tr.step, tr.loss, color=AZUL, linewidth=2, label="entrenamiento")
    if "eval_loss" in hist:
        ev = hist.dropna(subset=["eval_loss"])
        ax.plot(ev.step, ev.eval_loss, "o", color=NARANJA, markersize=8, markeredgecolor="white", label="validación")
    ax.set_xlabel("paso")
    ax.set_ylabel(ylabel)
    ax.set_title(titulo, loc="left", color=TINTA)
    ax.legend(frameon=False)
    plt.tight_layout()
    if guardar:
        fig.savefig(guardar, dpi=150)
    plt.show()

## 1. Índice del corpus por fuente

Se leen solo los pies de los parquets. Si un row group tiene una sola fuente, sus estadísticas (mín = máx) bastan; si mezcla fuentes, se lee solo la columna `source` de ese row group. El índice se guarda en `DIR_TRABAJO` y no se vuelve a calcular.

In [ ]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem()
RUTA_INDICE = DIR_TRABAJO / "indice_corpus.parquet"
HUB.bajar(RUTA_INDICE)
if RUTA_INDICE.exists():
    indice = pd.read_parquet(RUTA_INDICE)
else:
    archivos, fs = comun.listar_parquets(fs=fs)
    print(f"{len(archivos)} parquets; leyendo metadatos...")
    indice = comun.indice_corpus(archivos, fs)
    indice.to_parquet(RUTA_INDICE)
    HUB.subir(RUTA_INDICE)

rgs = indice.drop_duplicates(["archivo", "rg"])
print(f"Filas: {indice.filas.sum():,} | parquets: {indice.archivo.nunique()} | row groups: {len(rgs):,}")
print(f"Filas por row group (mediana): {rgs.filas_rg.median():,.0f} | MB por row group (mediana): {rgs.bytes_rg.median() / 1e6:.1f}")
print(f"Row groups con más de una fuente: {(indice.groupby(['archivo', 'rg']).size() > 1).sum():,}")

In [ ]:
fuentes = (indice.groupby("fuente")
           .agg(filas=("filas", "sum"), parquets=("archivo", "nunique"), row_groups=("rg", "size"))
           .sort_values("filas", ascending=False))
fuentes["%"] = (100 * fuentes.filas / fuentes.filas.sum()).round(2)

# ¿El corpus está ordenado por fuente? Rango de parquets en que aparece cada una
n_parquet = indice.archivo.str.extract(r"train-(\d+)-")[0].astype(int)
fuentes[["primer_parquet", "ultimo_parquet"]] = n_parquet.groupby(indice.fuente).agg(["min", "max"]).values
display(fuentes)
barras(fuentes.filas, "Textos por fuente (escala log)", "textos", log=True)

## 2. Muestra estratificada por fuente

Se eligen row groups al azar de toda la extensión del corpus hasta juntar `N_POR_FUENTE` textos de cada fuente. Es una muestra **por conglomerados** (textos vecinos en un row group pueden parecerse), suficiente para explorar, pero no para estimar proporciones finas.

In [ ]:
RUTA_MUESTRA = DIR_TRABAJO / f"muestra_{N_POR_FUENTE}.parquet"
HUB.bajar(RUTA_MUESTRA)
if RUTA_MUESTRA.exists():
    muestra = pd.read_parquet(RUTA_MUESTRA)
else:
    muestra = comun.muestra_por_fuente(indice, fs, N_POR_FUENTE, SEED)
    muestra.to_parquet(RUTA_MUESTRA)
    HUB.subir(RUTA_MUESTRA)
muestra["text"] = muestra.text.fillna("").astype(str)
ORDEN = fuentes.index.tolist()
print(muestra.source.value_counts())

In [ ]:
# Cinco textos al azar por fuente
for fuente in ORDEN:
    print(f"\n=== {fuente} ===")
    for t in muestra[muestra.source == fuente].text.sample(5, random_state=SEED):
        print("-", t[:300].replace("\n", " ⏎ "))

## 3. Longitudes y tokens de Gemma

In [ ]:
muestra["caracteres"] = muestra.text.str.len()
muestra["palabras"] = muestra.text.str.split().str.len()
display(muestra.groupby("source").palabras.describe(percentiles=[.1, .5, .9, .99]).round(0).loc[ORDEN])

fig, axes = plt.subplots(len(ORDEN), 1, figsize=(7, 1.5 * len(ORDEN) + 0.5), sharex=True)
axes = np.atleast_1d(axes)
bins = np.logspace(0, 4.5, 60)
for ax, f in zip(axes, ORDEN):
    ax.hist(muestra.loc[muestra.source == f, "palabras"].clip(1, 10 ** 4.5), bins=bins, color=AZUL)
    ax.set_xscale("log")
    ax.set_title(f, loc="left", fontsize=10, color=TINTA)
    ax.grid(axis="x", visible=False)
axes[-1].set_xlabel("palabras por texto (escala log)")
plt.tight_layout()
plt.show()

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(comun.MODELO_BASE)
muestra["tokens"] = [len(x) for x in tok(muestra.text.tolist(), add_special_tokens=False)["input_ids"]]

g = muestra.groupby("source")
tabla_tok = pd.DataFrame({
    "tokens_medios": g.tokens.mean(),
    "tokens_mediana": g.tokens.median(),
    "tokens_por_palabra": g.tokens.sum() / g.palabras.sum(),
    "filas_corpus": fuentes.filas,
})
tabla_tok["tokens_estimados_M"] = tabla_tok.tokens_medios * tabla_tok.filas_corpus / 1e6
display(tabla_tok.round(2).loc[ORDEN])
print(f"Total estimado: {tabla_tok.tokens_estimados_M.sum() / 1e3:,.2f} mil millones de tokens")
barras(tabla_tok.tokens_estimados_M, "Tokens de Gemma estimados por fuente (millones, escala log)", "millones de tokens", log=True)

## 4. Calidad: qué hay que limpiar

Porcentaje de textos de cada fuente con cada rasgo. `pocas_letras` marca textos donde menos del 60 % de los caracteres son letras (tablas, listas de precios, código).

In [ ]:
def proporcion(mascara):
    return mascara.groupby(muestra.source).mean().mul(100)


muestra["huella"] = muestra.text.map(comun.huella)
calidad = pd.DataFrame({
    "url_%": proporcion(muestra.text.str.contains(comun.RE_URL.pattern, regex=True)),
    "mencion_%": proporcion(muestra.text.str.contains(comun.RE_MENCION.pattern, regex=True)),
    "hashtag_%": proporcion(muestra.text.str.contains(comun.RE_HASHTAG.pattern, regex=True)),
    "retweet_%": proporcion(muestra.text.str.match(comun.RE_RT.pattern)),
    "menos_de_5_palabras_%": proporcion(muestra.palabras < 5),
    "pocas_letras_%": proporcion(~muestra.text.map(lambda t: comun.es_valido(t, min_palabras=0))),
    "mayusculas_%": proporcion(muestra.text.map(
        lambda t: sum(c.isupper() for c in t) / max(sum(c.isalpha() for c in t), 1) > 0.5)),
    "duplicados_%": muestra.groupby("source").huella.apply(lambda h: 100 * h.duplicated().mean()),
})
display(calidad.round(1).loc[ORDEN])

In [ ]:
# Comienzos más repetidos por fuente: revela plantillas, avisos legales, menús de sitios, bots
for fuente in ORDEN:
    inicios = muestra[muestra.source == fuente].text.str[:70].str.replace("\n", " ").value_counts().head(6)
    print(f"\n=== {fuente} ===")
    print(inicios.to_string())

In [ ]:
# Efecto de la limpieza que usará el paso 1 (comun.limpiar_texto + comun.es_valido + deduplicación)
muestra["limpio"] = muestra.text.map(lambda t: comun.limpiar_texto(t, MAX_CARACTERES))
muestra["valido"] = (muestra.limpio.map(lambda t: comun.es_valido(t, MIN_PALABRAS))
                     & ~muestra.limpio.map(comun.huella).duplicated())
efecto = pd.DataFrame({
    "sobrevive_%": proporcion(muestra.valido),
    "tokens_que_sobreviven_%": 100 * muestra[muestra.valido].groupby("source").tokens.sum()
                               / muestra.groupby("source").tokens.sum(),
})
display(efecto.round(1).loc[ORDEN])

for fuente in ORDEN:
    cambiados = muestra[(muestra.source == fuente) & (muestra.text != muestra.limpio)]
    if len(cambiados):
        r = cambiados.iloc[0]
        print(f"\n[{fuente}]\nANTES:   {r.text[:220]!r}\nDESPUÉS: {r.limpio[:220]!r}")

## 5. ¿Qué tan chilena y coloquial es cada fuente?

Dos medidas complementarias sobre los textos que sobreviven a la limpieza:

- **Marcadores coloquiales** (`comun.MARCADORES`): *po*, *cachai*, *weón*, voseo chileno (*estai*, *querís*), *al tiro*, *bacán/fome*, *pololo*, *carrete*, *cabros*, *harto*. Son pocos y muy específicos del habla chilena informal, así que funcionan como termómetro del registro.
- **Expresiones del diccionario** (las 802 de `rag_documentos.jsonl`). Cubren más, pero tienen ruido: palabras de una sola pieza como *once*, *pata* o *pato* también son español general.

In [ ]:
validos = muestra[muestra.valido].copy()
validos["palabras_limpio"] = validos.limpio.str.split().str.len()
palabras_fuente = validos.groupby("source").palabras_limpio.sum()

marc = pd.DataFrame([comun.contar_marcadores(t) for t in validos.limpio], index=validos.index)
dens = marc.groupby(validos.source).sum().div(palabras_fuente, axis=0).mul(1000)
dens.insert(0, "TOTAL_por_1000", dens.sum(axis=1))
dens.insert(1, "docs_con_marcador_%", (marc.sum(axis=1) > 0).groupby(validos.source).mean().mul(100))
display(dens.round(2).sort_values("TOTAL_por_1000", ascending=False))
barras(dens.TOTAL_por_1000, "Marcadores coloquiales chilenos por cada 1000 palabras", "por 1000 palabras", fmt="{:.2f}")

In [ ]:
from collections import Counter

det = comun.DetectorChilenismos(comun.ruta_dato(RAIZ, comun.RUTA_RAG))
MAX_PALABRAS_DOC = 400   # se revisan las primeras 400 palabras de cada texto
validos["expresiones"] = validos.limpio.map(lambda t: det.buscar(t, MAX_PALABRAS_DOC))
palabras_revisadas = validos.palabras_limpio.clip(upper=MAX_PALABRAS_DOC).groupby(validos.source).sum()

conteo = pd.DataFrame({f: Counter(x for l in g.expresiones for x in l)
                       for f, g in validos.groupby("source")}).fillna(0).astype(int)
expr = pd.DataFrame({
    "docs_con_expresion_%": (validos.expresiones.str.len() > 0).groupby(validos.source).mean().mul(100),
    "expresiones_por_1000": conteo.sum() / palabras_revisadas * 1000,
    "expresiones_distintas": (conteo > 0).sum(),
})
display(expr.round(2).loc[ORDEN])
for fuente in ORDEN:
    top = conteo[fuente].sort_values(ascending=False).head(25)
    print(f"\n{fuente}: " + ", ".join(f"{k} ({v})" for k, v in top.items() if v))

In [ ]:
# Expresiones sospechosas de ambigüedad: frecuentes en la fuente MENOS coloquial.
# Si son palabras de español general, conviene excluirlas del detector: comun.DetectorChilenismos(..., excluir={...})
tasa = conteo.div(palabras_revisadas, axis=1) * 1e5   # por 100 mil palabras
REF = dens.TOTAL_por_1000.idxmin()
COL = dens.TOTAL_por_1000.idxmax()
print(f"Fuente menos coloquial: {REF} | más coloquial: {COL}")
display(tasa.sort_values(REF, ascending=False).head(20).round(1))

# Expresiones características de la fuente más coloquial (razón de tasas, suavizada)
if COL != REF:
    razon = np.log2((tasa[COL] + 0.5) / (tasa[REF] + 0.5))
    print(f"\nMás propias de {COL} que de {REF} (log2 de la razón de tasas):")
    display(razon[conteo[COL] >= 5].sort_values(ascending=False).head(20).round(2))

## 6. ¿Cuántas expresiones de nuestro diccionario aparecen en el corpus?

Se buscan las 802 expresiones de `scraping/data/rag_documentos.jsonl` (clave, término y variantes, normalizados sin tildes ni puntuación) en el texto del corpus, no en la muestra. Para cada expresión se cuenta **en cuántos textos aparece**, por fuente, con un autómata Aho-Corasick repartido en varios procesos.

- `FRACCION_ESCANEO` es la fracción de row groups que se leen, al azar en todo el corpus. Los conteos se extrapolan al corpus completo. Con `0.05` se lee ~1 GB y tarda pocos minutos; con `1.0` se lee todo (~19 GB), lo que puede tomar ~1 h según los núcleos del entorno. Cada parquet deja su resultado en `DIR_TRABAJO/escaneo_diccionario/`, así que si la sesión se corta, al volver a ejecutar se retoma.
- Las expresiones de **una sola palabra** (*once*, *pato*, *micro*) también son español general y sus conteos están inflados. Las de **varias palabras** son una medida mucho más confiable.
- Que una expresión del split de prueba aparezca en el corpus no es fuga de datos: los pares de prueba son frases sintéticas que no están en el corpus.

Lo mismo se puede correr fuera del notebook: `python3 models/gemma4/contar_diccionario.py --fraccion 0.05`.

In [ ]:
FRACCION_ESCANEO = 0.05   # 1.0 = corpus completo

dic = comun.cargar_diccionario(RAIZ)
print(f"rag_documentos: {dic.clave.nunique()} expresiones, {dic.forma.nunique()} formas buscables")
HUB.bajar(DIR_TRABAJO / "escaneo_diccionario")   # retoma un escaneo cortado en otra sesión
docs_forma, filas_leidas = comun.escanear_diccionario(indice, dic.forma, DIR_TRABAJO / "escaneo_diccionario",
                                                      FRACCION_ESCANEO)
print(f"Textos leídos: {filas_leidas.sum():,} ({100 * filas_leidas.sum() / indice.filas.sum():.1f} % del corpus)")
presencia = comun.presencia_por_clave(dic, docs_forma, FRACCION_ESCANEO)
presencia.to_csv(DIR_TRABAJO / f"presencia_diccionario_f{FRACCION_ESCANEO}.csv")
HUB.subir(DIR_TRABAJO / "escaneo_diccionario")
HUB.subir(DIR_TRABAJO / f"presencia_diccionario_f{FRACCION_ESCANEO}.csv")
display(comun.resumen_presencia(presencia))

In [ ]:
# Distribución: cuántas expresiones caen en cada rango de textos (estimados en el corpus completo)
cortes = [-1, 0, 9, 99, 999, 9999, np.inf]
rangos = ["0", "1-9", "10-99", "100-999", "1.000-9.999", "10.000 o más"]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, (nombre, m) in zip(axes, [("de una palabra", ~presencia.varias_palabras),
                                  ("de varias palabras", presencia.varias_palabras)]):
    dist = pd.cut(presencia[m].textos_estimados, cortes, labels=rangos).value_counts().reindex(rangos)
    ax.bar(rangos, dist.values, color=AZUL, width=0.6)
    for x, v in enumerate(dist.values):
        ax.text(x, v, f"{v}", ha="center", va="bottom", color=TINTA_2, fontsize=9)
    ax.set_title(f"Expresiones {nombre} ({m.sum()})", loc="left", color=TINTA)
    ax.set_xlabel("textos del corpus que la contienen")
    ax.grid(axis="x", visible=False)
    ax.tick_params(axis="x", labelsize=8)
axes[0].set_ylabel("expresiones")
plt.tight_layout()
plt.show()

In [ ]:
cols = ["termino", "textos_estimados", *[c for c in presencia if c.startswith("textos_") and c != "textos_estimados"],
        "split_dataset"]
print("Más frecuentes (revisa las de una palabra: pueden ser español general):")
display(presencia.head(30)[cols])
print("Más frecuentes de varias palabras (las menos ambiguas):")
display(presencia[presencia.varias_palabras].head(20)[cols])
ausentes = presencia[presencia.textos_estimados == 0]
print(f"Sin apariciones ({len(ausentes)}):", ", ".join(ausentes.termino))

## 7. Mezcla propuesta para el paso 1

Regla por defecto: **peso de cada fuente proporcional a su densidad de marcadores coloquiales**, acotado entre `PESO_MIN` y `PESO_MAX`.

- `PESO_MIN` mantiene algo de español chileno estándar (noticias, web) para que el modelo no se desvíe del español general que después tiene que producir en la traducción.
- `PESO_MAX` evita que una sola plataforma domine el estilo.

La tabla compara los tokens pedidos con los disponibles tras la limpieza. **Revisa los resultados de arriba y, si hace falta, edita `PESOS` a mano** antes de guardar.

In [ ]:
PRESUPUESTO_TOKENS = 5_000_000   # ~30 min de entrenamiento en una A100 con LoRA en la FFN (20M: ~2 h)
PESO_MIN, PESO_MAX = 0.05, 0.60

pesos = comun.acotar_pesos(dens.TOTAL_por_1000, PESO_MIN, PESO_MAX)

plan = pd.DataFrame({
    "peso": pesos,
    "tokens_pedidos_M": pesos * PRESUPUESTO_TOKENS / 1e6,
    "tokens_disponibles_M": tabla_tok.tokens_estimados_M * efecto["tokens_que_sobreviven_%"] / 100,
})
plan["alcanza"] = plan.tokens_disponibles_M >= plan.tokens_pedidos_M
display(plan.round(3).sort_values("peso", ascending=False))

PESOS = pesos.round(4).to_dict()   # <- edita aquí si quieres otra mezcla, p. ej. {"tweets": 0.6, "news": 0.1, ...}

In [ ]:
config = {"pesos": PESOS, "presupuesto_tokens": PRESUPUESTO_TOKENS, "min_palabras": MIN_PALABRAS,
          "max_caracteres": MAX_CARACTERES, "seed": SEED, "n_por_fuente_muestra": N_POR_FUENTE}
with open(DIR_TRABAJO / "config_paso1.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
HUB.subir(DIR_TRABAJO / "config_paso1.json")
print(json.dumps(config, indent=2, ensure_ascii=False))

## Notas para interpretar

- **Muestra por conglomerados.** Los textos de un mismo row group suelen venir del mismo sitio o del mismo período, así que los porcentajes por fuente son aproximados. Si una fuente tiene pocos row groups, súbele `N_POR_FUENTE`.
- **Los marcadores son una heurística.** Miden registro coloquial, no "chilenidad" en general: una noticia de un medio chileno es chilena, pero no coloquial.
- **Ruido del detector.** Las expresiones de una palabra que también son español general inflan los conteos de las fuentes formales; revisa la tabla de expresiones sospechosas.
- **Datos personales.** El paso 1 reemplaza las menciones por `@usuario` y elimina las URLs. Los textos siguen la licencia CC-BY-SA 4.0 del corpus.